# GL Code Automation - UPS billing (CooperSurgical / Belimo)

1. Fill column M using the rules from the GL code logic doc (v1)
2. Compare with the completed files (answer key)
3. Look for patterns and write better rules (v2)

The final rules are also in `GL_Code_Apply.ipynb`, which is the one to run on new data.

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)


def col_idx(letter):
    # excel column letter -> 0-based index ('M' -> 12, 'AQ' -> 42)
    n = 0
    for ch in letter.strip().upper():
        n = n * 26 + (ord(ch) - 64)
    return n - 1


def read_ups_csv(path):
    # UPS files have no header row, read everything as text so nothing gets reformatted
    for enc in ["utf-8", "latin1"]:
        try:
            df = pd.read_csv(
                path, header=None, dtype=str, keep_default_na=False, encoding=enc
            )
            df.attrs["encoding"] = enc
            return df
        except UnicodeDecodeError:
            pass
    raise ValueError("could not read " + str(path))


def normalize_gl(code):
    # long dash -> '--', and 'Split 654...' / 'Split-654...' -> 'Split-654...'
    if not isinstance(code, str):
        return code
    code = code.strip()
    code = re.sub(r"[\u2014\u2015]", "--", code)
    code = re.sub(r"[\u2010\u2011\u2012\u2013]", "-", code)
    code = re.sub(r"^split[\s\-_:]*", "Split-", code, flags=re.I)
    return code


def get_col(df, c):
    return df[col_idx(c)].astype(str).str.strip()


def make_pattern(vals, word=False):
    if isinstance(vals, str):
        vals = [vals]
    alt = "|".join(re.escape(str(v)) for v in vals)
    if word:
        return r"(?<![A-Za-z0-9])(?:" + alt + r")(?![A-Za-z0-9])"
    return "(?:" + alt + ")"

In [ ]:
INPUT_DIR = Path("input")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


def detect_company(path):
    name = Path(path).name.upper()
    if "CSI" in name or "COOPER" in name:
        return "CSI"
    if "BELIMO" in name:
        return "Belimo"
    raise ValueError("can't tell the company from file name: " + Path(path).name)


def is_answer_key(path):
    name = Path(path).name
    return "_Completed_" in name and "Not_Completed" not in name


raw, truth = {}, {}
for p in sorted(INPUT_DIR.glob("*.csv")):
    comp = detect_company(p)
    if is_answer_key(p):
        truth[comp] = read_ups_csv(p)[col_idx("M")].map(normalize_gl).values
    else:
        raw[comp] = read_ups_csv(p)

for comp in raw:
    print(comp, raw[comp].shape, "| answer key:", comp in truth)

In [ ]:
# the completed file should be the same as the original except column M
for comp in raw:
    key = read_ups_csv(
        next(
            p
            for p in INPUT_DIR.glob("*.csv")
            if is_answer_key(p) and detect_company(p) == comp
        )
    )
    other = [c for c in raw[comp].columns if c != col_idx("M")]
    print(
        comp,
        "cells different outside column M:",
        int((key[other] != raw[comp][other]).to_numpy().sum()),
    )

## v1: rules from the doc

Notes on how I read the doc:
- GL codes end with two short dashes `--` (the long dash in the PDF is the same thing)
- doc says AQ = TAAX, the data only has TAX
- RMA / Lit use whole-word matching, otherwise "SHARMA" and "Facility" match
- SO55000 rules: the city check uses the sender city (BS) instead of CA. The 727 rows that were already filled in the CSI file all match with BS but only 30 match with CA
- Belimo "BP does not contain Warren or BP does not contain Belimo" is read as BP contains neither
- Belimo rules ending with "and Column CD" have no value for CD, so they are turned off

In [ ]:
# a rule is a list of conditions (col, op, value), all of them have to be true
# "P|Q" means column P or column Q
OPS = [
    "eq",
    "in",
    "contains",
    "word",
    "not_contains",
    "contains_any",
    "not_contains_any",
    "eq_col",
    "ne_col",
]


def cond_mask(df, cols, op, val):
    if op in ["eq_col", "ne_col"]:
        # compare two columns, e.g. sender city BS vs receiver city CA
        a = get_col(df, cols).str.lower()
        b = get_col(df, val).str.lower()
        return np.asarray(a == b) if op == "eq_col" else np.asarray(a != b)
    masks = []
    for c in cols.split("|"):
        s = get_col(df, c)
        if op == "eq":
            m = s.str.lower() == str(val).strip().lower()
        elif op == "in":
            m = s.str.lower().isin([str(v).strip().lower() for v in val])
        elif op == "word":
            m = s.str.contains(make_pattern(val, word=True), case=False, regex=True)
        else:  # contains / not_contains / contains_any / not_contains_any
            m = s.str.contains(make_pattern(val), case=False, regex=True)
        masks.append(np.asarray(m))
    if op.startswith("not_"):
        return ~np.logical_or.reduce(masks)
    return np.logical_or.reduce(masks)


def rule_mask(df, rule):
    m = np.ones(len(df), dtype=bool)
    for cols, op, val in rule["when"]:
        m &= cond_mask(df, cols, op, val)
    return m


def match_rules(df, rules):
    # for each row pick the matching rule with the highest priority
    # (ties -> the one listed first). conflict = top-priority rules disagree on the GL
    rules = [r for r in rules if r.get("enabled", True)]
    prio = np.array([r["priority"] for r in rules])
    gls = np.array([r["gl"] for r in rules], dtype=object)
    ids = np.array([r["id"] for r in rules], dtype=object)
    hits = np.column_stack([rule_mask(df, r) for r in rules])
    p = np.where(hits, prio, -1)
    best = p.max(axis=1)
    top = hits & (p == best[:, None])
    win = top.argmax(axis=1)
    matched = hits.any(axis=1)
    return pd.DataFrame(
        {
            "gl": np.where(matched, gls[win], ""),
            "rule": np.where(matched, ids[win], ""),
            "conflict": np.array([len(set(gls[row])) > 1 for row in top]),
            "all_rules": [";".join(ids[row]) for row in hits],
        }
    )

In [ ]:
SO_CITY = "BS"  # change to "CA" to follow the doc literally

RULES_V1 = {}
RULES_V1["CSI"] = [
    {
        "id": "CSI-01",
        "when": [("AQ", "eq", "SVC")],
        "gl": "65415-5500--CSI_U100-L529--",
    },
    {
        "id": "CSI-02",
        "when": [("AQ", "eq", "PRT")],
        "gl": "65415-5500--CSI_U100-L529--",
    },
    {
        "id": "CSI-03",
        "when": [("AQ", "eq", "AUD")],
        "gl": "65415-5500--CSI_U100-L529--",
    },
    {
        "id": "CSI-04",
        "when": [("AQ", "eq", "TAX")],
        "gl": "65420-3000--CSI_U100-L527--",
    },
    {
        "id": "CSI-05",
        "when": [("P", "eq", "83350-15")],
        "gl": "65415-8401--CSI_M100-L510--",
    },
    {
        "id": "CSI-06",
        "when": [("P", "eq", "83350-60")],
        "gl": "65415-6000--CSI_U100-L529--",
    },
    {
        "id": "CSI-07",
        "when": [("P", "eq", "83350-45")],
        "gl": "65415-1800--CSI_U100-L529--",
    },
    {
        "id": "CSI-08",
        "when": [("P", "eq", "83350-46")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-09",
        "when": [("P", "eq", "83350-80")],
        "gl": "65415-5500--CSI_U100-L529--",
    },
    {
        "id": "CSI-10",
        "when": [("AM", "eq", "helpdesk911")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-11",
        "when": [("AM", "eq", "BBryanCSI1")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-12",
        "when": [("AM", "eq", "lc963")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-13",
        "when": [("AM", "eq", "mcard636")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-14",
        "when": [("AM", "eq", "MThomasCSI1")],
        "gl": "65420-3000--CSI_U100-L529--",
    },
    {
        "id": "CSI-15",
        "when": [("AM", "eq", "rquin19")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-16",
        "when": [("AM", "eq", "Sheana Elliott")],
        "gl": "65420-3000--CSI_U100-L528--",
    },
    {
        "id": "CSI-17",
        "when": [("AM", "eq", "TMutambayi1")],
        "gl": "65420-3000--CSI_U100-L528--",
    },
    {
        "id": "CSI-18",
        "when": [("AM", "eq", "vdh510")],
        "gl": "65415-2000--CSI_M100-L529--",
    },
    {
        "id": "CSI-19",
        "when": [("BP", "contains", "Cooper"), ("BX", "contains", "Cooper")],
        "gl": "Cooper to Cooper",
    },
    {
        "id": "CSI-20",
        "when": [("BW", "contains", "repair")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-21",
        "when": [("BW", "word", "RMA")],
        "gl": "65420-3000--CSI_U100-L529--",
    },
    {
        "id": "CSI-22",
        "when": [("BW", "contains", "Returns")],
        "gl": "65420-3000--CSI_U100-L529--",
    },
    {
        "id": "CSI-23",
        "when": [("BW", "contains", "Complaint")],
        "gl": "65420-3000--CSI_U100-L529--",
    },
    {
        "id": "CSI-24",
        "when": [
            ("BP", "contains", "Cooper"),
            ("BS", "contains", "Trumb"),
            ("BX", "contains", "Cooper"),
            ("CA", "contains", "Alajuela"),
        ],
        "gl": "51630-1000--CSI_M100-L510--",
    },
    {
        "id": "CSI-25",
        "when": [
            ("BP", "contains", "Cooper"),
            ("BS", "contains", "Worth"),
            ("BX", "contains", "Cooper"),
            ("CA", "contains", "Alajuela"),
        ],
        "gl": "51630-1000--CSI_M100-L510--",
    },
    {
        "id": "CSI-26",
        "when": [("CA", "contains", "Tonawanda"), ("BP", "not_contains", "Cooper")],
        "gl": "65440-8401--CSI_P100-L551--",
    },
    {
        "id": "CSI-27",
        "when": [("CA", "contains", "Sugar Land"), ("BP", "not_contains", "Cooper")],
        "gl": "65440-8401--CSI_M100-L560--",
    },
    {
        "id": "CSI-28",
        "when": [("C", "eq", "0000H4B189"), ("CD", "eq", "CA")],
        "gl": "65420-3000--CSI_U100-L527--",
    },
    {
        "id": "CSI-29",
        "when": [("C", "eq", "0000H93A28")],
        "gl": "65420-3000--CSI_U100-L527--",
    },
    {
        "id": "CSI-30",
        "when": [("BW", "contains", "help desk")],
        "gl": "65415-1804--CSI_U100-L529--",
    },
    {
        "id": "CSI-31",
        "when": [("BW", "contains", "IT Dep")],
        "gl": "65415-1804--CSI_U100-L529--",
    },  # listed twice in the doc
    {
        "id": "CSI-32",
        "when": [("P|Q", "contains", "550-0"), ("CA", "contains", "Trumb")],
        "gl": "65440-8401--CSI_M100-L510--",
    },
    {
        "id": "CSI-33",
        "when": [("P|Q", "contains", "550-0"), ("CA", "contains", "Staff")],
        "gl": "65440-8401--CSI_M100-L562--",
    },
    {
        "id": "CSI-34",
        "when": [("P|Q", "contains", "550-0"), ("CA", "contains", "Worth")],
        "gl": "65440-8401--CSI_M100-L510--",
    },
    {
        "id": "CSI-35",
        "when": [("P", "contains", "Complaints")],
        "gl": "65415-1800--CSI_U100-L529--",
    },
    {
        "id": "CSI-36",
        "when": [("P", "contains", "Recall")],
        "gl": "65415-1800--CSI_U100-L529--",
    },
    {
        "id": "CSI-37",
        "when": [("P", "contains", "QA Dept")],
        "gl": "65415-8440--CSI_M100-L510--",
    },
    {
        "id": "CSI-38",
        "when": [
            ("P", "contains", "SO55000"),
            (SO_CITY, "contains", "Trumb"),
            ("BX", "not_contains", "Cooper"),
        ],
        "gl": "65420-3000--CSI_U100-L528--",
    },
    {
        "id": "CSI-39",
        "when": [
            ("P", "contains", "SO55000"),
            (SO_CITY, "contains", "Worth"),
            ("BX", "not_contains", "Cooper"),
        ],
        "gl": "65420-3000--CSI_U100-L527--",
    },
    {
        "id": "CSI-40",
        "when": [("BP", "contains", "Lynn Sales")],
        "gl": "65415-2113--CSI_U100-L529--",
    },
    {
        "id": "CSI-41",
        "when": [("BX", "contains", "Lynn Sales")],
        "gl": "65415-2113--CSI_U100-L529--",
    },
    {
        "id": "CSI-42",
        "when": [("CA", "contains", "Herten"), ("BP", "contains", "Cooper")],
        "gl": "51630-1000--CSI_M100-L510--",
    },
    # the last rule in the doc says "Column CA = SO55000", probably a typo for P, so it's the same as this one
    {
        "id": "CSI-43",
        "when": [
            ("P", "contains", "SO55000"),
            (SO_CITY, "contains", "Antonio"),
            ("BX", "not_contains", "Cooper"),
        ],
        "gl": "65420-3000--CSI_U100-L529--",
    },
]

RULES_V1["Belimo"] = [
    {"id": "BEL-01", "when": [("C", "contains", "9V1790")], "gl": "356200"},
    {"id": "BEL-02", "when": [("P", "contains", "sample")], "gl": "367211"},
    {"id": "BEL-03", "when": [("P", "contains", "Trade")], "gl": "367400"},
    {"id": "BEL-04", "when": [("P", "word", "Lit")], "gl": "367230"},
    {"id": "BEL-05", "when": [("P", "contains", "Promo")], "gl": "367230"},
    {"id": "BEL-06", "when": [("P", "word", "RMA")], "gl": "353590"},
    {"id": "BEL-07", "when": [("P", "contains", "2-10")], "gl": "367230"},
    {
        "id": "BEL-08",
        "when": [("BP", "contains", "Imprint"), ("BS", "contains", "Oshkosh")],
        "gl": "369600",
    },
    {
        "id": "BEL-09",
        "when": [("BY", "contains", "Imprint"), ("CA", "contains", "Oshkosh")],
        "gl": "369600",
    },  # BY is an address line, maybe meant BX
    {
        "id": "BEL-10",
        "when": [
            ("BP", "contains", "Belimo"),
            ("BS", "contains", "Danbury"),
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Sparks"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-11",
        "when": [
            ("BP", "contains", "Belimo"),
            ("BS", "contains", "Sparks"),
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Danbury"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-12",
        "when": [
            ("BP", "contains", "Warren"),
            ("BS", "contains", "Bethlehem"),
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Danbury"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-13",
        "when": [
            ("BP", "contains", "Warren"),
            ("BS", "contains", "Bethlehem"),
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Sparks"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-14",
        "when": [
            ("BP", "contains", "Belimo"),
            ("BS", "contains", "Danbury"),
            ("BX", "contains", "Warren"),
            ("CA", "contains", "Bethlehem"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-15",
        "when": [
            ("BP", "contains", "Belimo"),
            ("BS", "contains", "Sparks"),
            ("BX", "contains", "Warren"),
            ("CA", "contains", "Bethlehem"),
        ],
        "gl": "356205",
    },
    {
        "id": "BEL-16",
        "when": [
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Danbury"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "356200",
    },
    {
        "id": "BEL-17",
        "when": [
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Sparks"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "356200",
    },
    {
        "id": "BEL-18",
        "when": [
            ("BX", "contains", "Warren"),
            ("CA", "contains", "Bethlehem"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "356200",
    },
    {
        "id": "BEL-19",
        "enabled": False,
        "when": [
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Danbury"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "353550",
    },
    {
        "id": "BEL-20",
        "enabled": False,
        "when": [
            ("BX", "contains", "Belimo"),
            ("CA", "contains", "Sparks"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "353550",
    },
    {
        "id": "BEL-21",
        "enabled": False,
        "when": [
            ("BX", "contains", "Warren"),
            ("CA", "contains", "Bethlehem"),
            ("BP", "not_contains_any", ["Warren", "Belimo"]),
        ],
        "gl": "353550",
    },
    {"id": "BEL-22", "when": [("AJ", "contains", "ADC")], "gl": "353500"},
]

# if a row matches more than one rule, the one with more conditions wins (ties -> first in the doc)
# otherwise e.g. "Cooper to Cooper" would always beat the more specific Alajuela rule
for comp in RULES_V1:
    for r in RULES_V1[comp]:
        r["gl"] = normalize_gl(r["gl"])
        r["priority"] = len(r["when"])

In [ ]:
pred_v1 = {}
for comp in raw:
    res = match_rules(raw[comp], RULES_V1[comp])
    existing = raw[comp][col_idx("M")].map(normalize_gl).values
    gl = np.where(
        existing != "", existing, res["gl"].values
    )  # keep values that were already in M
    src = np.where(
        existing != "",
        "existing",
        np.where(res["gl"].values != "", "rule", "unmatched"),
    )
    pred_v1[comp] = pd.DataFrame({"gl": gl, "source": src, "rule": res["rule"].values})
    print(comp)
    print(pd.Series(src).value_counts().to_string(), "\n")

## v1 vs answer key

In [ ]:
for comp in pred_v1:
    p = pred_v1[comp]
    p["correct"] = p["gl"].values == truth[comp]
    print(comp, "overall accuracy: {:.1%}".format(p["correct"].mean()))
    print(p.groupby("source")["correct"].agg(["size", "mean"]).round(3), "\n")

In [ ]:
# for each doc rule: of the rows it matches, how many agree with the answer key?
def rule_audit(comp, rules):
    rows = []
    for r in rules:
        if not r.get("enabled", True):
            continue
        m = rule_mask(raw[comp], r)
        if m.sum() == 0:
            continue
        t = pd.Series(truth[comp][m])
        rows.append(
            {
                "id": r["id"],
                "gl": r["gl"],
                "matched": int(m.sum()),
                "agree": round((t == r["gl"]).mean(), 2),
                "most common answer": t.value_counts().index[0],
            }
        )
    return pd.DataFrame(rows).sort_values("agree")


for comp in raw:
    print(comp)
    display(rule_audit(comp, RULES_V1[comp]))

## Looking for patterns

Main idea: add a "flow" column based on whether the sender (BP) and receiver (BX) are the company itself.

In [ ]:
OWN = {"CSI": ["cooper"], "Belimo": ["belimo", "warren"]}

feat = {}
for comp in raw:
    df = raw[comp]
    f = pd.DataFrame(
        {
            c: get_col(df, c).values
            for c in [
                "C",
                "N",
                "P",
                "Q",
                "AJ",
                "AM",
                "AQ",
                "BP",
                "BS",
                "BW",
                "BX",
                "CA",
                "CD",
            ]
        }
    )
    pat = make_pattern(OWN[comp])
    sender_own = f["BP"].str.contains(pat, case=False, regex=True)
    receiver_own = f["BX"].str.contains(pat, case=False, regex=True)
    f["flow"] = np.select(
        [sender_own & receiver_own, sender_own, receiver_own],
        ["intercompany", "outbound", "inbound"],
        "third_party",
    )
    f["truth"] = truth[comp]
    feat[comp] = f

for comp in feat:
    print(comp)
    display(pd.crosstab(feat[comp]["truth"], feat[comp]["flow"], margins=True))

In [ ]:
def gl_profile(
    comp,
    gl,
    cols=["C", "AJ", "AQ", "AM", "BP", "BS", "BW", "BX", "CA", "CD", "P"],
    top=5,
):
    # most common values of each column for one GL code
    d = feat[comp][feat[comp]["truth"] == gl]
    out = {}
    for c in cols:
        vc = d[c].replace("", "<blank>").value_counts().head(top)
        out[c] = ", ".join(f"{k} ({v})" for k, v in vc.items())
    return pd.Series(out, name=f"{gl} ({len(d)} rows)").to_frame()


display(gl_profile("Belimo", "356205"))
display(gl_profile("Belimo", "369600"))
display(gl_profile("CSI", "51630-1000--CSI_M100-L510--"))

In [ ]:
# CSI inbound: receiving city decides the location part of the code
f = feat["CSI"]
inb = f[f["flow"] == "inbound"]
ct = pd.crosstab(inb["CA"].str.upper(), inb["truth"])
ct[ct.sum(axis=1) >= 4]

In [ ]:
# do adjustment lines (SCC, CHBK ...) have the same GL as the original shipment (same tracking #)?
for comp in feat:
    f = feat[comp]
    adj = f["AJ"].isin(["SCC", "CHBK", "RADJ", "ADC", "GCC"])
    orig = (
        f[~adj & (f["N"] != "")]
        .groupby("N")["truth"]
        .agg(lambda s: s.iloc[0] if s.nunique() == 1 else None)
        .dropna()
    )
    m = f.loc[adj, "N"].map(orig)
    found = m.notna()
    same = (m[found] == f.loc[adj, "truth"][found]).mean()
    print(
        f"{comp}: {adj.sum()} adjustment rows, {found.sum()} have the original in this file, same GL: {same:.1%}"
    )

### What I found

Both companies follow the same structure. Each layer below overrides the one above it:

0. **Default.** Outbound to customers. CSI = `65420-3000--CSI_U100-L527--` (DFW), Belimo = `353500`.
1. **Direction + site.**
   - Supplier to us: CSI `65440-8401` with the location from the receiving site (L510 / L560 / L562 / P100-L551); Belimo `356200`.
   - Between our own sites or affiliates: CSI `51630-1000` (Herten / Alajuela); Belimo `356205`.
   - Belimo export to a Canadian customer: `353550`. This is what the "and Column CD" rules meant (CD = CA).
2. **Charge type.** UPS service / audit / print charges = `65415-5500`; imports = `Split-65440-8401...`.
3. **Purpose.** Reference (P), attention (BW), UPS user (AM): complaints, IT, RMA, samples, trade shows, marketing.
4. **Adjustment lines** (SCC etc.) follow the original shipment with the same tracking #.

Places where the answer key doesn't match the doc:
- CSI: there is no "Cooper to Cooper" in the answers.
- CSI: P = 83350-15 is `65410-3000--CSI_U100-L529--`; user lc963 / mcard636 are `65415-5500`.
- CSI: Stafford purchases are L510 when the PO550 number is in P, and L562 when it's only in Q.
- CSI: BW "RMA ... RECALL" is a recall (`65415-1800`), not a return.
- Belimo: 2-10xxxx is an RMA number, so it's `353590`, not `367230`.
- Belimo trade shows: lit = `367230`, promo = `367210`, otherwise `367400`.
- Belimo: Belimo -> Warren is `353500`. Only Warren -> Belimo is `356205`.

## v2 rules

In [ ]:
OWN = {
    "CSI": ["cooper"],
    "Belimo": ["belimo", "warren"],
}  # words that mean "our company" in BP/BX

U527 = "65420-3000--CSI_U100-L527--"
U528 = "65420-3000--CSI_U100-L528--"
U529R = "65420-3000--CSI_U100-L529--"
COMPL = "65415-1800--CSI_U100-L529--"
IT = "65415-1804--CSI_U100-L529--"
SVC = "65415-5500--CSI_U100-L529--"
PUR = "65440-8401--CSI_M100-L510--"
ICO = "51630-1000--CSI_M100-L510--"
FROM_SUPPLIER = [
    ("BX", "contains_any", OWN["CSI"]),
    ("BP", "not_contains_any", OWN["CSI"]),
    ("AJ", "in", ["FC", "TP"]),
]

# layer: 0 default, 1-2 direction + site, 3 charge type, 4+ purpose (dept, person, reference)
# higher layer wins
RULES_V2 = {}
RULES_V2["CSI"] = [
    {
        "id": "C-DEF",
        "layer": 0,
        "when": [],
        "gl": U527,
        "why": "default: DFW outbound to customers",
    },
    {
        "id": "C-OUT-TRB",
        "layer": 1,
        "when": [("BP", "contains_any", OWN["CSI"]), ("BS", "contains", "Trumb")],
        "gl": U528,
        "why": "shipped from Trumbull",
    },
    {
        "id": "C-OUT-TON",
        "layer": 1,
        "when": [("BP", "contains_any", OWN["CSI"]), ("BS", "contains", "Tonawanda")],
        "gl": COMPL,
        "why": "shipped from North Tonawanda (complaints/recalls)",
    },
    {
        "id": "C-IN-RET",
        "layer": 1,
        "when": [
            ("BX", "contains_any", OWN["CSI"]),
            ("BP", "not_contains_any", OWN["CSI"]),
        ],
        "gl": COMPL,
        "why": "sent to Cooper by outside party, assume customer return",
    },
    {
        "id": "C-IN-PUR-TRB",
        "layer": 2,
        "when": FROM_SUPPLIER + [("CA", "contains_any", ["Trumb", "Worth"])],
        "gl": PUR,
        "why": "supplier to Trumbull/DFW",
    },
    {
        "id": "C-IN-PUR-SGL",
        "layer": 2,
        "when": FROM_SUPPLIER + [("CA", "contains", "Sugar Land")],
        "gl": "65440-8401--CSI_M100-L560--",
        "why": "supplier to Sugar Land",
    },
    {
        "id": "C-IN-PUR-STF",
        "layer": 2,
        "when": FROM_SUPPLIER + [("CA", "contains", "Stafford")],
        "gl": "65440-8401--CSI_M100-L562--",
        "why": "supplier to Stafford",
    },
    {
        "id": "C-IN-PUR-STF-PO",
        "layer": 3,
        "when": FROM_SUPPLIER
        + [("CA", "contains", "Stafford"), ("P", "contains", "550-0")],
        "gl": PUR,
        "why": "Stafford, PO550 number in P (Trumbull PO)",
    },
    {
        "id": "C-IN-PUR-STF-3P",
        "layer": 2,
        "when": [("BX", "contains", "LONESTAR"), ("CA", "contains", "Stafford")],
        "gl": "65440-8401--CSI_M100-L562--",
        "why": "supplier to Stafford (LoneStar)",
    },
    {
        "id": "C-IN-PUR-TON",
        "layer": 2,
        "when": FROM_SUPPLIER + [("CA", "contains", "Tonawanda")],
        "gl": "65440-8401--CSI_P100-L551--",
        "why": "supplier to North Tonawanda",
    },
    {
        "id": "C-IMP",
        "layer": 2,
        "when": [("BX", "contains_any", OWN["CSI"]), ("AJ", "eq", "IMP")],
        "gl": "Split-" + PUR,
        "why": "import -> split",
    },
    {
        "id": "C-ICO-INTL",
        "layer": 2,
        "when": [("BX", "contains_any", OWN["CSI"]), ("CD", "in", ["NL", "CR"])],
        "gl": ICO,
        "why": "to Cooper entity abroad (Herten / Alajuela)",
    },
    {
        "id": "C-SVC",
        "layer": 3,
        "when": [("AQ", "in", ["SVC", "AUD", "PRT"])],
        "gl": SVC,
        "why": "UPS service / audit / print charge",
    },
    {
        "id": "C-TAX",
        "layer": 3,
        "when": [("AQ", "eq", "TAX")],
        "gl": U527,
        "why": "tax",
    },
    {
        "id": "C-SO-TRB",
        "layer": 4,
        "when": [("P", "contains", "SO55000"), ("BS", "contains", "Trumb")],
        "gl": U528,
        "why": "Trumbull sales order",
    },
    {
        "id": "C-SO-DFW",
        "layer": 4,
        "when": [("P", "contains", "SO55000"), ("BS", "contains", "Worth")],
        "gl": U527,
        "why": "DFW sales order (incl. overseas distributor)",
    },
    {
        "id": "C-IT-ACCT",
        "layer": 4,
        "when": [("C", "eq", "0000XH8253")],
        "gl": IT,
        "why": "IT UPS account",
    },
    {
        "id": "C-IT-BW",
        "layer": 4,
        "when": [("BW", "contains_any", ["IT Dep", "help desk", "repair"])],
        "gl": IT,
        "why": "IT dept / repair",
    },
    {
        "id": "C-IT-AM",
        "layer": 4,
        "when": [("AM", "in", ["helpdesk911", "BBryanCSI1", "rquin19", "EdMak"])],
        "gl": IT,
        "why": "IT user id",
    },
    {
        "id": "C-P-1804",
        "layer": 4,
        "when": [("P", "eq", "83350-46")],
        "gl": IT,
        "why": "reference 83350-46",
    },
    {
        "id": "C-SVC-AM",
        "layer": 4,
        "when": [("AM", "in", ["mcard636", "lc963", "A.MollicaCSI"])],
        "gl": SVC,
        "why": "user id (answer key says 5500, doc said 1804)",
    },
    {
        "id": "C-P-5500",
        "layer": 4,
        "when": [("P", "eq", "83350-80")],
        "gl": SVC,
        "why": "reference 83350-80",
    },
    {
        "id": "C-RMA",
        "layer": 4,
        "when": [("BW", "word", ["RMA", "Returns", "Complaint"])],
        "gl": U529R,
        "why": "RMA / returns dept",
    },
    {
        "id": "C-RMA-RECALL",
        "layer": 5,
        "when": [("BW", "contains", "RECA")],
        "gl": COMPL,
        "why": "RMA ... RECALL = recall",
    },
    {
        "id": "C-RMA-AM",
        "layer": 4,
        "when": [("AM", "in", ["MThomasCSI1"])],
        "gl": U529R,
        "why": "returns user id",
    },
    {
        "id": "C-L528-AM",
        "layer": 4,
        "when": [("AM", "in", ["Sheana Elliott", "TMutambayi1"])],
        "gl": U528,
        "why": "user id",
    },
    {
        "id": "C-COMPL",
        "layer": 4,
        "when": [("P", "contains_any", ["Complaint", "Recall", "83350-45"])],
        "gl": COMPL,
        "why": "complaint / recall",
    },
    {
        "id": "C-QA",
        "layer": 5,
        "when": [("P", "contains", "QA Dept")],
        "gl": "65415-8440--CSI_M100-L510--",
        "why": "QA dept",
    },
    {
        "id": "C-P-6000",
        "layer": 4,
        "when": [("P", "eq", "83350-60")],
        "gl": "65415-6000--CSI_U100-L529--",
        "why": "reference 83350-60",
    },
    {
        "id": "C-P-15",
        "layer": 4,
        "when": [("P", "eq", "83350-15")],
        "gl": "65410-3000--CSI_U100-L529--",
        "why": "answer key differs from doc (doc: 65415-8401--CSI_M100-L510--)",
    },
    {
        "id": "C-SHOW",
        "layer": 4,
        "when": [
            (
                "AM|BW|BX",
                "contains_any",
                ["vdh510", "Tara Luczak", "Maria Tobin", "Drake Hotel", "AUGS"],
            )
        ],
        "gl": "65415-2000--CSI_M100-L529--",
        "why": "trade show / conference",
    },
    {
        "id": "C-LYNN",
        "layer": 4,
        "when": [("BP|BX", "contains", "Lynn Sales")],
        "gl": "65415-2113--CSI_U100-L529--",
        "why": "AL-Lynn Sales rep",
    },
]

RULES_V2["Belimo"] = [
    {
        "id": "B-DEF",
        "layer": 0,
        "when": [],
        "gl": "353500",
        "why": "default: outbound to customers",
    },
    {
        "id": "B-OUT-CA",
        "layer": 1,
        "when": [("CD", "eq", "CA"), ("BX", "not_contains_any", OWN["Belimo"])],
        "gl": "353550",
        "why": "export to Canada (not an affiliate)",
    },
    {
        "id": "B-IN",
        "layer": 1,
        "when": [
            ("BX", "contains_any", OWN["Belimo"]),
            ("BP", "not_contains_any", OWN["Belimo"]),
        ],
        "gl": "356200",
        "why": "supplier to Belimo",
    },
    {
        "id": "B-ICO",
        "layer": 2,
        "when": [
            ("BX", "contains", "belimo"),
            ("BP", "contains_any", OWN["Belimo"]),
            ("BS", "ne_col", "CA"),
        ],
        "gl": "356205",
        "why": "between Belimo sites / affiliates, or Warren to Belimo",
    },
    {
        "id": "B-MKT",
        "layer": 3,
        "when": [("BP", "contains", "imprint")],
        "gl": "369600",
        "why": "marketing: 4imprint",
    },
    {
        "id": "B-MKT2",
        "layer": 3,
        "when": [
            ("BW|P", "contains_any", ["Marissa", "BUSINES CARD", "Field Support"])
        ],
        "gl": "369600",
        "why": "marketing people / business cards",
    },
    {
        "id": "B-RMA",
        "layer": 4,
        "when": [("P", "word", "RMA")],
        "gl": "353590",
        "why": "RMA",
    },
    {
        "id": "B-RMA2",
        "layer": 4,
        "when": [("P", "contains", "2-10")],
        "gl": "353590",
        "why": "2-10xxxx is an RMA number (doc said 367230)",
    },
    {
        "id": "B-SAMPLE",
        "layer": 4,
        "when": [("P", "contains", "sample")],
        "gl": "367211",
        "why": "samples",
    },
    {
        "id": "B-TRADE",
        "layer": 4,
        "when": [("P", "contains", "trade")],
        "gl": "367400",
        "why": "trade show",
    },
    {
        "id": "B-PROMO",
        "layer": 5,
        "when": [("P", "contains", "trade"), ("P", "contains", "promo")],
        "gl": "367210",
        "why": "trade show promo",
    },
    {
        "id": "B-LIT",
        "layer": 6,
        "when": [("P", "word", "lit")],
        "gl": "367230",
        "why": "trade show literature",
    },
]

# priority = layer * 10 + number of conditions
for comp in RULES_V2:
    for r in RULES_V2[comp]:
        r["gl"] = normalize_gl(r["gl"])
        r["priority"] = r["layer"] * 10 + len(r["when"])
        for cols, op, val in r["when"]:
            assert op in OPS, r["id"]
print({k: len(v) for k, v in RULES_V2.items()})

In [ ]:
# adjustment-type lines (shipping charge corrections etc.) follow the original shipment
ADJUSTMENT_TYPES = ["SCC", "CHBK", "RADJ", "ADC", "GCC"]
PROTECTED_RULES = ["C-SVC", "C-TAX"]  # these keep their own GL


def inherit_from_tracking(raw, gl, rule, ledger=None):
    gl = gl.copy()
    rule = rule.copy()
    trk = get_col(raw, "N").values
    adj = np.isin(get_col(raw, "AJ").values, ADJUSTMENT_TYPES)

    base = pd.DataFrame({"N": trk, "GL": gl, "R": rule})[~adj & (trk != "")]
    src_rule = base.drop_duplicates("N").set_index("N")["R"]
    if ledger is not None and len(ledger) > 0:
        base = pd.concat([base, ledger[["N", "GL"]]])
    n_gl = base.groupby("N")["GL"].nunique()
    one = (
        base[base["N"].isin(n_gl[n_gl == 1].index)]
        .drop_duplicates("N")
        .set_index("N")["GL"]
    )

    new_gl = pd.Series(trk).map(one).values
    ok = adj & pd.notna(new_gl) & ~np.isin(rule, PROTECTED_RULES)
    gl[ok] = new_gl[ok]
    rule[ok] = "TRACKING"
    from_rule = np.where(ok, pd.Series(trk).map(src_rule).fillna("history").values, "")
    multi = adj & pd.Series(trk).isin(n_gl[n_gl > 1].index).values
    return gl, rule, from_rule, multi

In [ ]:
pred_v2 = {}
for comp in raw:
    res = match_rules(raw[comp], RULES_V2[comp])
    gl, rule, from_rule, multi = inherit_from_tracking(
        raw[comp], res["gl"].values.astype(object), res["rule"].values.astype(object)
    )
    pred_v2[comp] = pd.DataFrame({"gl": gl, "rule": rule, "inherited_from": from_rule})

summary = []
for comp in raw:
    summary.append(
        {
            "company": comp,
            "v1 accuracy": round((pred_v1[comp]["gl"].values == truth[comp]).mean(), 4),
            "v2 accuracy": round((pred_v2[comp]["gl"].values == truth[comp]).mean(), 4),
            "v2 wrong rows": int((pred_v2[comp]["gl"].values != truth[comp]).sum()),
        }
    )
pd.DataFrame(summary)

In [ ]:
# accuracy per rule and the rows that are still wrong
for comp in raw:
    p = pred_v2[comp].copy()
    p["correct"] = p["gl"].values == truth[comp]
    print(comp)
    display(
        p.groupby("rule")["correct"].agg(["size", "mean"]).sort_values("mean").head(10)
    )

    wrong = feat[comp][~p["correct"].values].assign(
        pred=p["gl"][~p["correct"]].values, rule=p["rule"][~p["correct"]].values
    )
    display(
        wrong.groupby(["rule", "pred", "truth", "AJ", "BP", "BX"])
        .size()
        .rename("rows")
        .sort_values(ascending=False)
        .head(15)
        .reset_index()
    )
    wrong.to_csv(OUTPUT_DIR / f"{comp}_v2_wrong_rows.csv", index=False)

Notes:
- v2 was written while looking at this answer key, so the accuracy is in-sample and probably too high. The real test is the next batch.
- Most of the rows that are still wrong look the same as other rows but have a different answer, e.g. CSI IT shipments where the SCC correction line got 65415-1003. I need to ask accounting about these.
- A few rules use people's names (Marissa Palmer, Tara Luczak) and will need updating if people change.